# Chapter 07 — Intelligence in the Wrong Direction

**Companion to Applied AI**

Question: How can we distinguish an apparent model improvement from statistical noise?

By the end of this notebook you will have:

- compared two stochastic systems with repeated samples
- computed means, standard errors, and paired estimates
- shown why 14-6 on twenty tasks may tell us very little

## What this notebook demonstrates
A synthetic evaluation harness. Nothing here evaluates a real model; it shows the *statistics* of small evaluations so you stop over-reading them.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import math

seed: 42


## 1. Single run vs repeated runs

In [2]:
def system_score(rng: random.Random, p: float = 0.65) -> int:
    return 1 if rng.random() < p else 0

N_TASKS, N_RUNS = 40, 5
runs = [[system_score(random.Random(SEED + r * 1000 + t)) for t in range(N_TASKS)] for r in range(N_RUNS)]
means = [sum(r) / N_TASKS for r in runs]
print("single-run scores:", [round(m, 2) for m in means])
import statistics
print(f"mean over runs: {statistics.mean(means):.3f}, spread: {min(means):.2f}-{max(means):.2f}")
assert max(means) - min(means) > 0, "repeated runs should vary"

single-run scores: [0.72, 0.65, 0.62, 0.95, 0.57]
mean over runs: 0.705, spread: 0.57-0.95


## 2. Mean, standard error, confidence interval

In [3]:
scores = runs[0]
p_hat = sum(scores) / N_TASKS
se = math.sqrt(p_hat * (1 - p_hat) / N_TASKS)
print(f"p_hat={p_hat:.3f}  SE={se:.3f}  approx 95% CI=[{p_hat-1.96*se:.3f}, {p_hat+1.96*se:.3f}]")
experiment = {"seed": SEED, "n": N_TASKS, "runs": N_RUNS}
print("settings:", experiment)

p_hat=0.725  SE=0.071  approx 95% CI=[0.587, 0.863]
settings: {'seed': 42, 'n': 40, 'runs': 5}


## 3. The famous 14-6 on twenty tasks

In [4]:
def sign_test_p(k: int, n: int) -> float:
    from math import comb
    return sum(comb(n, i) for i in range(k, n + 1)) / 2**n

p = sign_test_p(14, 20)
print(f"14/20 wins under H0 (equally good): one-sided p = {p:.3f}")
print("Verdict:", "not significant at 0.05" if p > 0.05 else "significant")
assert p > 0.05

14/20 wins under H0 (equally good): one-sided p = 0.058
Verdict: not significant at 0.05


## 4. Paired comparison beats unpaired averages

In [5]:
rng = random.Random(SEED)
tasks = 30
# paired: same tasks, B slightly better on hard ones
a = [1 if rng.random() < 0.6 else 0 for _ in range(tasks)]
b = [1 if (rng.random() < (0.75 if av == 0 else 0.55)) else 0 for av in a]
wins = sum(1 for x, y in zip(a, b) if y > x)
losses = sum(1 for x, y in zip(a, b) if x > y)
ties = tasks - wins - losses
print(f"paired on {tasks} tasks: B wins {wins}, A wins {losses}, ties {ties}")
wa = sum(a)/tasks; wb = sum(b)/tasks
print(f"unpaired means: A={wa:.2f} B={wb:.2f} (gap {wb-wa:+.2f}); paired view shows where the gap lives")

paired on 30 tasks: B wins 9, A wins 9, ties 12
unpaired means: A=0.63 B=0.63 (gap +0.00); paired view shows where the gap lives


## Interpretation
- Supports: repeated runs + SE + paired tests keep you honest; 14–6/20 (p≈0.12) does not establish superiority.
- Does NOT support: any claim about real systems A or B.

## Try it yourself
1. Grow the task set from 20 to 200 at the same win rate and watch p shrink.
2. Remove pairing (shuffle B's scores) and compare the SE.
3. Find the smallest win margin on 20 tasks that reaches p < 0.05.